# Task 3: Cleaning Data

**Intern:** Malaika Saleem  
**Track:** Data Analytics  
**Program:** Oasis Infobyte Internship (OIBSIP)  
**Dataset:** Titanic passenger data (`tested.csv`)

## Objective

Clean the dataset step by step: handle missing values, remove duplicates, fix formatting, detect outliers, correct data types, and save a clean CSV.

In [10]:
import pandas as pd

df = pd.read_csv("tested.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


## Step 1: Data Quality Report

Before cleaning, I checked the dataset for problems.

- **Shape:** 418 rows and 12 columns
- **Duplicate rows:** 0
- **Missing values:** Age (86), Fare (1), Cabin (327)

In [11]:
print(f"Shape: {df.shape}")
print(f"Duplicate rows: {df.duplicated().sum()}")
print()
print("Missing values per column:")
print(df.isnull().sum())
print()
df.info()

Shape: (418, 12)
Duplicate rows: 0

Missing values per column:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Survived     418 non-null    int64  
 2   Pclass       418 non-null    int64  
 3   Name         418 non-null    str    
 4   Sex          418 non-null    str    
 5   Age          332 non-null    float64
 6   SibSp        418 non-null    int64  
 7   Parch        418 non-null    int64  
 8   Ticket       418 non-null    str    
 9   Fare         417 non-null    float64
 10  Cabin        91 non-null     str    
 11  Embarked     418 non-null    str    
dtypes: float

## Step 2: Handling Missing Values

- **Age (86 missing):** filled with the median, because the median is not affected by extreme ages.
- **Fare (1 missing):** filled with the median, because only one value was missing.
- **Cabin (327 missing):** dropped, because about 78% of the values were empty and guessing them would be incorrect.

In [12]:
df_raw = df.copy()

df["Age"] = df["Age"].fillna(df["Age"].median())
df["Fare"] = df["Fare"].fillna(df["Fare"].median())
df = df.drop(columns="Cabin")

print(df.isnull().sum())

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64


## Step 3: Duplicates and Formatting

- **Duplicates:** I checked for duplicate rows and 0 were found, so none were removed.
- **Formatting:** I checked the Sex and Embarked columns and their values were already consistent, so no changes were needed.

In [13]:
before = len(df)
df = df.drop_duplicates()
print(f"Duplicates removed: {before - len(df)}")

print(df["Sex"].value_counts())
print(df["Embarked"].value_counts())
print(df["Age"].isnull().sum())

Duplicates removed: 0
Sex
male      266
female    152
Name: count, dtype: int64
Embarked
S    270
C    102
Q     46
Name: count, dtype: int64
0


## Step 4: Detecting Outliers

I used the IQR method to find values far away from the rest of the data.

- **Age:** 36 outliers found (limits: 3.88 to 54.88)
- **Fare:** 55 outliers found (upper limit: 66.84)

In [14]:
for col in ["Age", "Fare"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"{col}: {len(outliers)} outliers (limits: {lower:.2f} to {upper:.2f})")

Age: 36 outliers (limits: 3.88 to 54.88)
Fare: 55 outliers (limits: -27.47 to 66.84)


## Step 5: Handling Outliers

- **Age:** outliers were kept, because they are real ages (babies and older passengers), not errors.
- **Fare:** high values were capped at the upper limit (66.84) in a new column, `Fare_capped`, because a few very expensive tickets can distort the average.
- The original `Fare` column was kept unchanged.

In [15]:
q1 = df["Fare"].quantile(0.25)
q3 = df["Fare"].quantile(0.75)
upper = q3 + 1.5 * (q3 - q1)

df["Fare_capped"] = df["Fare"].clip(upper=upper)

print(f"Fare max before: {df['Fare'].max()}")
print(f"Fare_capped max after: {df['Fare_capped'].max():.2f}")

Fare max before: 512.3292
Fare_capped max after: 66.84


## Step 6: Correcting Data Types

- **PassengerId:** changed from number (int) to text (str), because it is only an identifier and we do not do calculations on it.
- **Age and Fare:** kept as float, because they contain decimal values.

In [16]:
df["PassengerId"] = df["PassengerId"].astype("str")

df.dtypes

PassengerId        str
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Embarked           str
Fare_capped    float64
dtype: object

## Step 7: Before vs After Summary

This table compares the raw data with the cleaned data.

- **Null values:** reduced from 414 to 0.
- **Columns:** still 12, because Cabin was dropped and Fare_capped was added.
- **PassengerId:** changed from int64 to str.

In [17]:
summary = pd.DataFrame({
    "Metric": ["Rows", "Columns", "Total null values", "Duplicate rows", "PassengerId dtype"],
    "Before": [len(df_raw), df_raw.shape[1], df_raw.isnull().sum().sum(),
               df_raw.duplicated().sum(), str(df_raw["PassengerId"].dtype)],
    "After": [len(df), df.shape[1], df.isnull().sum().sum(),
              df.duplicated().sum(), str(df["PassengerId"].dtype)]
})
summary

,Metric,Before,After
0,Rows,418,418
1,Columns,12,12
2,Total null values,414,0
3,Duplicate rows,0,0
4,PassengerId dtype,int64,str


## Step 8: Saving the Cleaned Data

The cleaned dataset was saved as `tested_cleaned.csv`.

**Summary:** missing values were handled, no duplicates were found, outliers were reviewed, and data types were corrected.

In [18]:
df.to_csv("tested_cleaned.csv", index=False)
print("Cleaned file saved")

Cleaned file saved
